In [1]:
import os
import tiktoken
import outlines
import outlines.models as models

# outlines内部使用tiktoken获取tokenizer，需要为qwen模型注册兼容的编码器
tiktoken.model.MODEL_TO_ENCODING["qwen-max"] = "cl100k_base"
tiktoken.model.MODEL_TO_ENCODING["qwen-turbo"] = "cl100k_base"

examples = [
    {"question": "37593 * 67 等于多少?", "code": "37593 * 67"},
    {
        "question": "小红的鸭子每天下16个蛋。她每天早餐吃3个，每天用4个做蛋糕给朋友。她把剩下的鸭蛋以每个2元的价格在市场上卖掉。她每天在市场上能赚多少钱?",
        "code": "(16-3-4)*2",
    },
    {
        "question": "做一件长袍需要2匹蓝色布料和一半数量的白色布料。总共需要多少匹布料?",
        "code": "2 + 2/2",
    },
]

question = "小明正在下载一个200GB的文件。他的下载速度是2GB/分钟，但下载到40%时下载失败了。然后小明不得不从头开始重新下载。他总共花了多少分钟下载完文件?"


@outlines.prompt
def answer_with_code_prompt(question, examples):
    """
    {% for example in examples %}
    QUESTION: {{example.question}}
    CODE: {{example.code}}

    {% endfor %}
    QUESTION: {{question}}
    CODE:"""

def extract_code(text):
    """从模型返回的文本中提取可执行的代码表达式"""
    import re
    # 尝试提取CODE:后面的代码
    match = re.search(r'CODE:\s*(.+)', text)
    if match:
        code_line = match.group(1).strip()
        # 去掉行尾注释
        code_line = re.split(r'\s*#', code_line)[0].strip()
        return code_line
    # 如果没有CODE:前缀，取第一行非空内容
    for line in text.strip().split('\n'):
        line = line.strip()
        if line and not line.startswith('#'):
            return re.split(r'\s*#', line)[0].strip()
    return text.strip()


prompt = answer_with_code_prompt(question, examples)

# 使用DashScope API(兼容OpenAI协议)
model_name = "qwen-max"
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"

model = models.openai(model_name, api_key=api_key, base_url=base_url)

answer = outlines.generate.text(model)(prompt)
print('answer=', answer)
code = extract_code(answer)
print('code=', code)
result = eval(code)
print(f"下载文件总共花费 {result:.0f} 分钟")


answer= 根据题目描述，小明下载一个200GB的文件，当下载进度达到40%时失败，之后不得不再次从头开始下载直至完成。我们先计算两次下载过程中所花费的时间总和。

- 文件大小为200GB。
- 下载速度为2GB/分钟。
- 第一次下载时，下载了文件的40%，即\(200GB \times 40\% = 80GB\)。
- 由于下载速度是2GB/分钟，因此第一次下载80GB所需时间为\(80GB \div 2GB/分钟 = 40\)分钟。
- 第二次重新下载整个200GB文件所需时间为\(200GB \div 2GB/分钟 = 100\)分钟。

因此，小明总共花费的时间为第一次下载时间加上第二次下载时间，即\(40分钟 + 100分钟 = 140\)分钟。

所以，最终答案是小明总共花费了140分钟来下载完文件。

CODE: 40 + 100 # 将两次下载所耗时间相加得到总时间
code= 40 + 100
下载文件总共花费 140 分钟
